# Counterfactuals Calculation  
## Getting CFs based on user's preferences


In [11]:
# ============================================================
# 0) Libraries
# ============================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

import dice_ml
from dice_ml import Dice

In [3]:
# ============================================================
# 1) Config
# ============================================================

CSV_PATH = Path("surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv")
ART_DIR  = Path("surrogate_pkl_cfs_metadata")
MODEL_PATH = ART_DIR / "surrogate_catboost.cbm"

TARGET = "value"

FEATURES = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_coreset_sampling_ratio",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_num_neighbors",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

CATEGORICAL = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
]

SOFT_TRAIN_FRACTION_RANGE = (0.20, 1.00)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]  # 0.005..0.5

In [4]:
# ============================================================
# 2) Small IO + prompt helpers
# ============================================================

def pick(prompt, options):
    while True:
        print("\n" + prompt)
        for i, opt in enumerate(options, start=1):
            print(f"  {i}) {opt}")
        s = input("Choose: ").strip()
        if s.isdigit():
            k = int(s)
            if 1 <= k <= len(options):
                return k - 1
        print("Invalid choice. Please type a number from the list.")


def ask_float(prompt, lo, hi, step=None):
    while True:
        s = input(f"{prompt} [{lo}..{hi}]: ").strip().replace(",", ".")
        try:
            x = float(s)
            if x < lo or x > hi:
                print("Out of range.")
                continue
            if step is not None:
                x = round((x - lo) / step) * step + lo
                x = max(lo, min(hi, x))
                x = float(f"{x:.6f}")
            return x
        except ValueError:
            print("Invalid number. Try again.")


def snap_budget(x):
    return min(SOFT_REVIEW_BUDGETS, key=lambda b: abs(b - x))


def show_factual(row, cols):
    return ", ".join([f"{c}={row[c]}" for c in cols])

In [5]:
# ============================================================
# 3) Load dataset + surrogate
# ============================================================

assert CSV_PATH.is_file(), f"Missing dataset: {CSV_PATH.resolve()}"
assert MODEL_PATH.is_file(), f"Missing surrogate model: {MODEL_PATH.resolve()}"

df = pd.read_csv(CSV_PATH)

for c in FEATURES + [TARGET]:
    assert c in df.columns, f"Missing column in dataset: {c}"

# Cast categoricals to str for stability (DiCE + CatBoost)
for c in CATEGORICAL:
    df[c] = df[c].astype(str)

cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

print("Loaded df:", df.shape)
print("Loaded surrogate:", MODEL_PATH)

Loaded df: (2999, 14)
Loaded surrogate: surrogate_pkl_cfs_metadata/surrogate_catboost.cbm


In [1]:
# ============================================================
# 4) Pick factual
# ============================================================

rng = np.random.default_rng(42)
idxs = rng.choice(df.index.to_numpy(), size=3, replace=False)

show_cols = [
    "params_backbone", "params_image_size_key", "params_layers_key",
    "params_soft_corruption_level", "params_soft_review_budget", "params_soft_train_fraction",
]

print("\n=== Pick a factual configuration (starting point) ===\n")
cands = []
for i, idx in enumerate(idxs, start=1):
    row = df.loc[idx, FEATURES + [TARGET]]
    print(f"[{i}] value={row[TARGET]:.6f} | " + show_factual(row, show_cols))
    cands.append((idx, row))

choice = pick("Select factual (1, 2, or 3):", ["Factual 1", "Factual 2", "Factual 3"])
selected_idx, selected_row = cands[choice]

x_factual = selected_row[FEATURES].copy()
y_factual_true = float(selected_row[TARGET])

x_factual_df = pd.DataFrame([x_factual.values], columns=FEATURES)
for c in CATEGORICAL:
    x_factual_df[c] = x_factual_df[c].astype(str)

y_factual_sur = float(cb.predict(x_factual_df)[0])

print("\nSelected factual index:", int(selected_idx))
print(f"Selected factual true value={y_factual_true:.6f}, surrogate_pred={y_factual_sur:.6f}")

display(x_factual_df)

NameError: name 'np' is not defined

In [7]:
# ============================================================
# 5) User objective + soft constraints (for CFs only)
# ============================================================

print("\n=== Soft knobs (what they mean) ===")
print("soft_train_fraction = data effort / training compute")
print("soft_review_budget  = human effort at deployment")
print("soft_corruption_level = robustness regularization")
print("Performance (value) = surrogate-predicted quality\n")

goal_type_idx = pick(
    "Performance goal:",
    [
        "Reach at least a minimum F1 (value >= target)",
        "Improve by a delta over the factual (value >= factual + delta)",
        "Stay within a target band [L, U]",
    ],
)

objective = {"type": None}
if goal_type_idx == 0:
    t = ask_float("Enter target_min F1", 0.0, 1.0)
    objective.update({"type": "target_min", "target_min": t})
elif goal_type_idx == 1:
    d = ask_float("Enter delta improvement (e.g., 0.03)", 0.0, 1.0)
    objective.update({"type": "delta_improve", "delta": d})
else:
    L = ask_float("Enter band lower bound L", 0.0, 1.0)
    U = ask_float("Enter band upper bound U", L, 1.0)
    objective.update({"type": "target_band", "target_band": [L, U]})


soft_constraints = {}

# soft_train_fraction
mode_idx = pick(
    "Constraint for soft_train_fraction (applies to CFs only):",
    ["No constraint (free)", "Fix to a value", "Restrict to a range"],
)
if mode_idx == 0:
    soft_constraints["params_soft_train_fraction"] = {"mode": "free"}
elif mode_idx == 1:
    v = ask_float("Fix soft_train_fraction to", SOFT_TRAIN_FRACTION_RANGE[0], SOFT_TRAIN_FRACTION_RANGE[1], step=0.01)
    soft_constraints["params_soft_train_fraction"] = {"mode": "fixed", "value": float(v)}
else:
    lo = ask_float("Range lower bound", SOFT_TRAIN_FRACTION_RANGE[0], SOFT_TRAIN_FRACTION_RANGE[1], step=0.01)
    hi = ask_float("Range upper bound", lo, SOFT_TRAIN_FRACTION_RANGE[1], step=0.01)
    soft_constraints["params_soft_train_fraction"] = {"mode": "range", "bounds": [float(lo), float(hi)]}

# soft_review_budget
mode_idx = pick(
    "Constraint for soft_review_budget (applies to CFs only):",
    ["No constraint (free)", "Fix to a value", "Restrict to a range"],
)
if mode_idx == 0:
    soft_constraints["params_soft_review_budget"] = {"mode": "free"}
elif mode_idx == 1:
    v = ask_float("Fix soft_review_budget to", min(SOFT_REVIEW_BUDGETS), max(SOFT_REVIEW_BUDGETS), step=0.005)
    v = snap_budget(v)
    soft_constraints["params_soft_review_budget"] = {"mode": "fixed", "value": float(v)}
else:
    lo = ask_float("Range lower bound", min(SOFT_REVIEW_BUDGETS), max(SOFT_REVIEW_BUDGETS), step=0.005)
    hi = ask_float("Range upper bound", lo, max(SOFT_REVIEW_BUDGETS), step=0.005)
    lo, hi = snap_budget(lo), snap_budget(hi)
    soft_constraints["params_soft_review_budget"] = {"mode": "range", "bounds": [float(lo), float(hi)]}

# soft_corruption_level
mode_idx = pick(
    "Constraint for soft_corruption_level (applies to CFs only):",
    ["No constraint (free)", "Fix to factual value", "Choose allowed levels"],
)
if mode_idx == 0:
    soft_constraints["params_soft_corruption_level"] = {"mode": "free"}
elif mode_idx == 1:
    soft_constraints["params_soft_corruption_level"] = {
        "mode": "fixed",
        "value": str(x_factual["params_soft_corruption_level"]),
    }
else:
    print("\nSelect allowed corruption levels (type 1/2/3 separated by spaces, e.g. '1 3'):")
    for i, lvl in enumerate(SOFT_CORRUPTION_LEVELS, start=1):
        print(f"  {i}) {lvl}")
    while True:
        s = input("Allowed: ").strip()
        parts = [p for p in s.split() if p.isdigit()]
        chosen = sorted(set(int(p) for p in parts if 1 <= int(p) <= 3))
        if chosen:
            allowed = [SOFT_CORRUPTION_LEVELS[i - 1] for i in chosen]
            break
        print("Invalid. Example: 1 2 3")
    soft_constraints["params_soft_corruption_level"] = {"mode": "allowed", "allowed": allowed}

k = int(["3", "5", "8"][pick("How many counterfactual options to return?", ["3", "5", "8"])])
rank_mode = ["best", "cheapest", "balanced"][pick("Selection mode:", ["best performance", "cheapest effort", "balanced"])]

query = {
    "factual": {
        "index": int(selected_idx),
        "x": x_factual.to_dict(),
        "value_true": y_factual_true,
        "value_surrogate": y_factual_sur,
    },
    "objective": objective,
    "soft_constraints": soft_constraints,
    "selection": {"k": int(k), "mode": rank_mode},
}

print("\n=== Query ===")
print(json.dumps(query, indent=2))


=== Soft knobs (what they mean) ===
soft_train_fraction = data effort / training compute
soft_review_budget  = human effort at deployment
soft_corruption_level = robustness regularization
Performance (value) = surrogate-predicted quality


Performance goal:
  1) Reach at least a minimum F1 (value >= target)
  2) Improve by a delta over the factual (value >= factual + delta)
  3) Stay within a target band [L, U]


Choose:  3
Enter band lower bound L [0.0..1.0]:  0.75
Enter band upper bound U [0.75..1.0]:  0.95



Constraint for soft_train_fraction (applies to CFs only):
  1) No constraint (free)
  2) Fix to a value
  3) Restrict to a range


Choose:  1



Constraint for soft_review_budget (applies to CFs only):
  1) No constraint (free)
  2) Fix to a value
  3) Restrict to a range


...


How many counterfactual options to return?
  1) 3
  2) 5
  3) 8


Choose:  3



Selection mode:
  1) best performance
  2) cheapest effort
  3) balanced


Choose:  2



=== Query ===
{
  "factual": {
    "index": 2320,
    "x": {
      "params_backbone": "resnet34",
      "params_batch_size": "16",
      "params_center_crop_key": "0.875",
      "params_coreset_sampling_ratio": 0.099,
      "params_image_size_key": "224",
      "params_layers_key": "l3",
      "params_max_patches_per_image": "1024",
      "params_num_neighbors": 2,
      "params_pre_trained": "True",
      "params_reduction": "mean",
      "params_soft_corruption_level": "none",
      "params_soft_review_budget": 0.365,
      "params_soft_train_fraction": 0.89
    },
    "value_true": 0.8978930307941653,
    "value_surrogate": 0.8905744295099763
  },
  "objective": {
    "type": "target_band",
    "target_band": [
      0.75,
      0.95
    ]
  },
  "soft_constraints": {
    "params_soft_train_fraction": {
      "mode": "free"
    },
    "params_soft_review_budget": {
      "mode": "free"
    },
    "params_soft_corruption_level": {
      "mode": "free"
    }
  },
  "selection": {
   

In [8]:
# ============================================================
# 6) Translate objective -> desired_range
# ============================================================

obj = query["objective"]
factual_base = float(query["factual"]["value_surrogate"])  # baseline for delta

if obj["type"] == "target_min":
    L, U = float(obj["target_min"]), 1.0
elif obj["type"] == "delta_improve":
    L, U = factual_base + float(obj["delta"]), 1.0
elif obj["type"] == "target_band":
    L, U = map(float, obj["target_band"])
else:
    raise ValueError(f"Unknown objective type: {obj['type']}")

desired_range = [max(0.0, L), min(1.0, U)]
print("desired_range:", desired_range)

desired_range: [0.75, 0.95]


In [9]:
# ============================================================
# 7) DiCE setup + permitted_range (Option C) + generate CFs
# ============================================================

# DiCE training dataframe
df_dice = df[FEATURES + [TARGET]].copy()

CONTINUOUS = [c for c in FEATURES if c not in CATEGORICAL]
data_dice = dice_ml.Data(dataframe=df_dice, continuous_features=CONTINUOUS, outcome_name=TARGET)
model_dice = dice_ml.Model(model=cb, backend="sklearn", model_type="regressor")
exp_dice = Dice(data_dice, model_dice, method="random")

# factual to DiCE
x0_dice = x_factual_df[FEATURES].copy()

def domain_for_feature(feat: str):
    if feat in CATEGORICAL:
        return sorted(df_dice[feat].dropna().astype(str).unique().tolist())
    col = pd.to_numeric(df_dice[feat], errors="coerce")
    return [float(np.nanmin(col)), float(np.nanmax(col))]

def factual_value(feat: str):
    v = x0_dice.iloc[0][feat]
    return str(v) if feat in CATEGORICAL else float(v)

def safe_range(lo, hi, feat):
    try:
        lo = float(lo); hi = float(hi)
        if lo > hi:
            return None
        dom_lo, dom_hi = domain_for_feature(feat)
        lo = max(dom_lo, lo)
        hi = min(dom_hi, hi)
        if lo > hi:
            return None
        return [float(lo), float(hi)]
    except Exception:
        return None

def safe_allowed(allowed, feat):
    try:
        dom = set(map(str, domain_for_feature(feat)))
        allowed = [str(a) for a in allowed]
        allowed = [a for a in allowed if a in dom]
        return sorted(set(allowed)) if allowed else None
    except Exception:
        return None

# Build permitted_range from soft_constraints.
# If invalid/missing -> default to factual fixed.
permitted_range = {}
for feat, spec in query["soft_constraints"].items():
    mode = spec.get("mode", "free")

    if mode == "free":
        permitted_range[feat] = domain_for_feature(feat)

    elif mode == "fixed":
        if feat in CATEGORICAL:
            v = str(spec.get("value", factual_value(feat)))
            ok = safe_allowed([v], feat) or safe_allowed([factual_value(feat)], feat)
            permitted_range[feat] = ok
        else:
            v = spec.get("value", factual_value(feat))
            r = safe_range(v, v, feat) or safe_range(factual_value(feat), factual_value(feat), feat)
            permitted_range[feat] = r

    elif mode == "range":
        if feat in CATEGORICAL:
            # invalid for categoricals -> factual fixed
            permitted_range[feat] = safe_allowed([factual_value(feat)], feat)
        else:
            bounds = spec.get("bounds", None)
            if not bounds or len(bounds) != 2:
                permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)
            else:
                r = safe_range(bounds[0], bounds[1], feat)
                permitted_range[feat] = r if r is not None else safe_range(factual_value(feat), factual_value(feat), feat)

    elif mode == "allowed":
        if feat not in CATEGORICAL:
            permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)
        else:
            allowed = spec.get("allowed", None)
            ok = safe_allowed(allowed, feat) if allowed is not None else None
            permitted_range[feat] = ok if ok is not None else safe_allowed([factual_value(feat)], feat)

    else:
        # unknown mode -> factual fixed
        if feat in CATEGORICAL:
            permitted_range[feat] = safe_allowed([factual_value(feat)], feat)
        else:
            permitted_range[feat] = safe_range(factual_value(feat), factual_value(feat), feat)

# Remove None entries
permitted_range = {k: v for k, v in permitted_range.items() if v is not None}

print("\npermitted_range (constraints injected into DiCE search):")
for k_, v_ in permitted_range.items():
    print(f" - {k_}: {v_}")

pool = int(max(20, 5 * query["selection"]["k"]))
print("\nGenerating pool:", pool)
print("desired_range:", desired_range)

dice_res = exp_dice.generate_counterfactuals(
    x0_dice,
    total_CFs=pool,
    desired_range=desired_range,
    features_to_vary=FEATURES,
    permitted_range=permitted_range,
)


permitted_range (constraints injected into DiCE search):
 - params_soft_train_fraction: [0.2, 1.0]
 - params_soft_review_budget: [0.005, 0.5]
 - params_soft_corruption_level: ['mild', 'none', 'strong']

Generating pool: 40
desired_range: [0.75, 0.95]


100%|██████████| 1/1 [00:00<00:00,  4.65it/s]


In [10]:
# ============================================================
# 8) Final CF table: minimal safety band + rank + top-k
# ============================================================

ex = dice_res.cf_examples_list[0]
cf_raw = ex.final_cfs_df_sparse.copy() if hasattr(ex, "final_cfs_df_sparse") else ex.final_cfs_df.copy()

assert TARGET in cf_raw.columns, f"DiCE CF table missing '{TARGET}'. Columns: {list(cf_raw.columns)}"

cf_df = cf_raw[FEATURES + [TARGET]].copy()
for c in CATEGORICAL:
    if c in cf_df.columns:
        cf_df[c] = cf_df[c].astype(str)

# safety: enforce desired band
L, U = desired_range
cf_df = cf_df[(cf_df[TARGET] >= L) & (cf_df[TARGET] <= U)].copy()

# rank and take top-k (using DiCE 'value' only)
k = int(query["selection"]["k"])
cf_df = cf_df.sort_values(TARGET, ascending=False).reset_index(drop=True).head(k)

display(cf_df)
print(f"Kept {len(cf_df)} CFs (requested {k}).")

,params_backbone,params_batch_size,params_center_crop_key,params_coreset_sampling_ratio,params_image_size_key,params_layers_key,params_max_patches_per_image,params_num_neighbors,params_pre_trained,params_reduction,params_soft_corruption_level,params_soft_review_budget,params_soft_train_fraction,value
0,resnet34,16,0.875,0.099,224,l3,1024,2,True,mean,none,0.365,0.65,0.894497
1,resnet34,16,0.875,0.099,224,l3,1024,2,True,mean,none,0.344,0.89,0.890821
2,resnet34,16,0.875,0.098,224,l3,1024,2,True,mean,none,0.365,0.89,0.890283
3,resnet34,16,0.875,0.093,224,l3,1024,2,True,mean,none,0.365,0.89,0.890222
4,resnet34,16,0.875,0.099,256,l3,1024,2,True,mean,none,0.365,0.89,0.889050
5,resnet34,16,0.875,0.099,224,l3,1024,2,True,mean,none,0.379,0.89,0.888470
6,resnet34,16,0.875,0.078,224,l3,1024,2,True,mean,none,0.365,0.89,0.884431
7,resnet34,16,0.875,0.074,224,l3,1024,2,True,mean,none,0.365,0.89,0.884431


Kept 8 CFs (requested 8).


In [13]:
# ============================================================
# 9) EXPORT
# ============================================================

EXPORT_DIR = Path("surrogate_pkl_cfs_metadata")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

cfs_out = cf_df.copy()
cfs_out.insert(0, "cf_id", range(len(cfs_out)))
cfs_out = cfs_out[["cf_id"] + FEATURES + [TARGET]]
cfs_out.to_csv(EXPORT_DIR / "cfs.csv", index=False)

meta = {
    "surrogate_model_path": str(MODEL_PATH),
    "splits_dir": "patchcore_splits",
    "target": TARGET,
    "features": FEATURES,
    "categorical": CATEGORICAL,
    "factual_index": int(query["factual"]["index"]),
    "factual_value_surrogate": float(query["factual"]["value_surrogate"]),
    "objective": query["objective"],
    "soft_constraints": query["soft_constraints"],
    "selection": query["selection"],
    "desired_range": desired_range,
    "permitted_range": permitted_range,
    "n_cfs": int(len(cfs_out)),
}
with open(EXPORT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Saved:", EXPORT_DIR / "cfs.csv")
print("Saved:", EXPORT_DIR / "metadata.json")


Saved: surrogate_pkl_cfs_metadata/cfs.csv
Saved: surrogate_pkl_cfs_metadata/metadata.json
